In [314]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time
import copy
import math
import networkx as nx
import ipycytoscape

In [315]:
def load_circuit_from_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    N = int(lines[0].strip())

    s, t, E = lines[1].split()
    s = int(s)
    t = int(t)
    E = float(E)
    CG = nx.Graph()
    for i in range(N):
        CG.add_node(i,point=i)
    for v, u, r in [tuple(l.split()) for l in lines[2:]]:
        CG.add_edge(int(v), int(u), R=float(r), G=1/float(r), I = 0.0)

    return CG, s, t, E

def show_circuit(CG):
    cytoscapeobj = ipycytoscape.CytoscapeWidget()
    cytoscapeobj.graph.add_graph_from_networkx(CG)
    cytoscapeobj.set_style([
    {
        'selector': 'node',
        'css': {
            'content': 'data(point)', 
            'text-valign': 'center',
            'color': 'black',
            'background-color': 'lightgray'
        }
    },
    {
        'selector': 'edge',
        'css': {
            'content': 'data(R)', 
            'font-size': '12px',
            'text-background-color': 'white',
            'text-background-opacity': 0.8
        }
    }
    ])
    display(cytoscapeobj)

def show_solved_circuit(CG):
    vis_graph = nx.DiGraph()
    for n, d in CG.nodes(data=True):
        vis_graph.add_node(n, **d)
    max_I = max([abs(d.get('I', 0)) for _, _, d in CG.edges(data=True)], default=1)
    if max_I == 0: 
        max_I = 1 
        
    for u, v, d in CG.edges(data=True):
        I = round(d.get('I', 0), 3)
        
        node_u = max(u, v) 
        node_v = min(u, v)
        
        if I > 0:
            src, tgt = node_v, node_u
        elif I < 0:
            src, tgt = node_u, node_v
        else:
            src, tgt = node_v, node_u 
            
        edge_data = d.copy()
        edge_data["I"] = I
        edge_data['abs_I'] = abs(I)
        
        vis_graph.add_edge(src, tgt, **edge_data)

    cytoscapeobj = ipycytoscape.CytoscapeWidget()
    cytoscapeobj.graph.add_graph_from_networkx(vis_graph)
    
    cytoscapeobj.set_style([
        {
            'selector': 'node',
            'css': {
                'content': 'data(point)', 
                'text-valign': 'center',
                'color': 'black',
                'background-color': 'lightgray'
            }
        },
        {
            'selector': 'edge',
            'css': {
                'content': 'data(I)', 
                'font-size': '12px',
                'text-background-color': 'white',
                'text-background-opacity': 0.8,
                
                'curve-style': 'bezier', 
                'target-arrow-shape': 'triangle',
                
                'line-color': f'mapData(abs_I, 0, {max_I}, #a1d99b, #005a32)', 
                'target-arrow-color': f'mapData(abs_I, 0, {max_I}, #a1d99b, #005a32)',
                'width': f'mapData(abs_I, 0, {max_I}, 1.5, 6)',
                'opacity': f'mapData(abs_I, 0, {max_I}, 0.3, 1.0)'
            }
        }
    ])
    
    display(cytoscapeobj)

def print_matrix(X):
    for row in X:
        for x in row:
            print(round(x,2), end='\t')
        print()

In [316]:

def add_circuit_parameters(G):
    nx.set_edge_attributes(G, 0, 'I')
    nx.set_edge_attributes(G, 1, 'R')
    nx.set_edge_attributes(G, 1, 'G')
    nx.set_node_attributes(G, dict([i, i] for i in range(len(G.nodes))), 'point' )

def random_erdos(N, max_tries = 50):
    p = (math.log(N) + 0.01*N)/N
    tries = 0
    while not nx.is_connected(G := nx.erdos_renyi_graph(N,p)) and tries < max_tries: tries += 1
    add_circuit_parameters(G)
    return G

def random_double_erdos(N, max_tries = 50):
    if N % 2 == 1: N+=1
    G1 = random_erdos(N//2)
    G2 = random_erdos(N//2)
    G = nx.disjoint_union(G1, G2)
    G.add_edge(0, N-1)
    add_circuit_parameters(G)
    return G

def grid_2d(N):
    a = math.ceil(math.sqrt(N))    
    G = nx.grid_2d_graph(a, a)
    add_circuit_parameters(G)
    return G

def cubical():
    G = nx.cubical_graph
    add_circuit_parameters(G)

    return G
def small_world(N):
    k = N//5
    p = 0.05
    G = nx.watts_strogatz_graph(N,k,p)
    add_circuit_parameters(G)
    return G


In [317]:
 

def nodal_analysis(CGin : nx.Graph, s : int, t : int, E : float):

    '''
    params:
    G -> networkx graph representing a circuit
    s, t -> two nodes with a voltage of E between them
    E -> voltage
    '''
    CG = CGin.copy()
    N = len(CG.nodes)
    X = np.zeros((N-2,N-2))
    Y = np.zeros(N-2)

    unk = list(filter(lambda i : i != s and i != t, range(N)))

    for i in range(N-2):
        for j in range(N-2):
            if i == j:
                X[i][i]= sum(list(map(lambda edge : CG[edge[0]][edge[1]]["G"], CG.edges(unk[i]))))
            else:
                edge_data = CG.get_edge_data(unk[i], unk[j], default=0)
                X[i][j] = 0 if edge_data == 0 else -edge_data["G"]

    for i in range(N-2):
        edge_data = CG.get_edge_data(unk[i], s, default=0)
        Y[i] = 0 if edge_data == 0 else -edge_data["G"]*E 
    
    Vunk = list(np.linalg.solve(X, Y))
    V = []
    if s < t:
        V = Vunk[0:s] + [-E] + Vunk[s:t] + [0] + Vunk[t:]
    else:
        V = Vunk[0:t] + [0] + Vunk[t:s] + [-E] + Vunk[s:]

    print(len(V))
    print(N)
    for i in range(N):
        for j in range(i+1, N):
            if (i,j) not in CG.edges: continue
            U = V[j]-V[i]
            I = float(CG[i][j]["G"] * abs(U))
            CG[i][j]["I"] = I*np.sign(U)
    
    return CG            



In [318]:
#CG,s,t,E = load_circuit_from_file("testgraph")
CG = random_erdos(6)
CG = nodal_analysis(CG,0,2,0.1)
show_circuit(CG)
show_solved_circuit(CG)

6
6


CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'content': 'd…

CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'content': 'd…